# Лабораторная работа 8.2  
## Анализ bias и аудит модели

Тема: выявление, измерение и документирование bias в ML-модели, расчёт fairness-метрик и подготовка Model Card.

В ноутбуке реализован полный цикл:

- генерация демонстрационного датасета кредитного скоринга;
- выбор sensitive attribute;
- EDA;
- preprocessing;
- обучение baseline-модели;
- расчёт стандартных метрик качества;
- групповой анализ качества модели;
- расчёт fairness-метрик через Fairlearn;
- mitigation bias через `ThresholdOptimizer`;
- сравнение baseline и mitigated-модели;
- сохранение графиков;
- генерация отчёта и Model Card.

## 1. Подготовка структуры проекта

Создаём каталоги, соответствующие требованиям лабораторной работы.

In [ ]:
from pathlib import Path

project_dirs = [
    "data",
    "notebooks",
    "src",
    "reports",
    "reports/figures",
]

for directory in project_dirs:
    Path(directory).mkdir(parents=True, exist_ok=True)

print("Структура проекта создана.")

## 2. Файл зависимостей

Для выполнения ноутбука потребуются:

- `pandas`;
- `numpy`;
- `scikit-learn`;
- `matplotlib`;
- `seaborn`;
- `fairlearn`;
- `jupyter`.

AIF360 в данном решении не используется, так как для базовой реализации достаточно Fairlearn.

In [ ]:
Path("requirements.txt").write_text(
    '''pandas==2.2.3
numpy==2.2.1
scikit-learn==1.6.0
matplotlib==3.10.0
seaborn==0.13.2
fairlearn==0.12.0
jupyter==1.1.1
''',
    encoding="utf-8",
)

print("requirements.txt создан.")

## 3. Импорт библиотек

Импортируем основные библиотеки для анализа данных, обучения модели, оценки качества и fairness-аудита.

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from fairlearn.metrics import (
    MetricFrame,
    selection_rate,
    demographic_parity_difference,
    demographic_parity_ratio,
    equalized_odds_difference,
    true_positive_rate,
    false_positive_rate,
    false_negative_rate,
)
from fairlearn.postprocessing import ThresholdOptimizer

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

DATA_DIR = Path("data")
REPORTS_DIR = Path("reports")
FIGURES_DIR = REPORTS_DIR / "figures"

print("Библиотеки импортированы.")

## 4. Генерация демонстрационного датасета

Используется синтетический датасет кредитного скоринга.

Целевая переменная:

- `target = 1` — заявка одобрена;
- `target = 0` — заявка не одобрена.

Sensitive attribute:

- `sex`: `Male` / `Female`.

Важно: датасет создаётся так, чтобы в исторических данных присутствовало смещение.  
Это нужно для демонстрации fairness-аудита и mitigation bias.

In [ ]:
def generate_credit_scoring_dataset(n_samples: int = 5000, random_state: int = 42) -> pd.DataFrame:
    # Генерирует демонстрационный датасет кредитного скоринга.
    #
    # В данных намеренно присутствует исторический bias:
    # при прочих равных условиях группа Female получает немного меньшую вероятность
    # положительной метки в исторических данных.
    #
    # Это учебная имитация, а не реальный датасет.

    rng = np.random.default_rng(random_state)

    sex = rng.choice(["Male", "Female"], size=n_samples, p=[0.55, 0.45])
    age = rng.integers(18, 70, size=n_samples)

    # Доход частично зависит от возраста и пола, что создаёт proxy-эффект.
    base_income = rng.normal(65000, 22000, size=n_samples)
    income = (
        base_income
        + age * 450
        + np.where(sex == "Male", 4500, -2500)
    )
    income = np.clip(income, 12000, 220000)

    employment_years = np.clip(
        (age - 18) * rng.uniform(0.25, 0.75, size=n_samples)
        + rng.normal(0, 3, size=n_samples),
        0,
        45,
    )

    loan_amount = rng.normal(180000, 85000, size=n_samples)
    loan_amount = np.clip(loan_amount, 10000, 650000)

    education = rng.choice(
        ["school", "bachelor", "master"],
        size=n_samples,
        p=[0.35, 0.45, 0.20],
    )

    region = rng.choice(
        ["urban", "suburban", "rural"],
        size=n_samples,
        p=[0.50, 0.30, 0.20],
    )

    credit_history_years = np.clip(
        employment_years + rng.normal(2, 4, size=n_samples),
        0,
        50,
    )

    debt_to_income = loan_amount / income

    # Базовый скоринг.
    score = (
        -0.2
        + 0.000025 * income
        - 0.85 * debt_to_income
        + 0.045 * employment_years
        + 0.035 * credit_history_years
        + np.where(education == "master", 0.35, 0)
        + np.where(education == "bachelor", 0.15, 0)
        + np.where(region == "urban", 0.10, 0)
        + np.where(region == "rural", -0.10, 0)
    )

    # Исторический bias в метке.
    # Это моделирует ситуацию, когда прошлые решения были не полностью справедливыми.
    score += np.where(sex == "Female", -0.35, 0.0)

    probability_approval = 1 / (1 + np.exp(-score))
    target = rng.binomial(1, probability_approval)

    df = pd.DataFrame(
        {
            "age": age,
            "sex": sex,
            "income": income.round(2),
            "loan_amount": loan_amount.round(2),
            "employment_years": employment_years.round(2),
            "credit_history_years": credit_history_years.round(2),
            "education": education,
            "region": region,
            "debt_to_income": debt_to_income.round(4),
            "target": target,
        }
    )

    df["age_group"] = np.where(df["age"] >= 25, "age >= 25", "age < 25")

    return df


df = generate_credit_scoring_dataset()

data_path = DATA_DIR / "sample_data.csv"
df.to_csv(data_path, index=False)

print(f"Датасет сохранён: {data_path}")
print(df.head())
print(df.shape)

## 5. Описание датасета

Сохраним краткое описание датасета в `data/README.md`.

In [ ]:
Path("data/README.md").write_text(
    '''# Demo Credit Scoring Dataset

Датасет является синтетическим и используется только для учебной лабораторной работы.

## Назначение

Задача: бинарная классификация кредитной заявки.

- `target = 1` — заявка одобрена.
- `target = 0` — заявка не одобрена.

## Sensitive attributes

Основной sensitive attribute:

- `sex`: Male / Female.

Дополнительный потенциальный sensitive attribute:

- `age_group`: age >= 25 / age < 25.

## Важное замечание

В данные намеренно встроен исторический bias, чтобы можно было выполнить fairness-аудит и применить mitigation bias.
''',
    encoding="utf-8",
)

print("data/README.md создан.")

## 6. Первичный анализ данных

Проверим:

- размер датасета;
- типы признаков;
- пропуски;
- распределение target;
- распределение sensitive attribute;
- долю положительного класса по группам.

In [ ]:
print("Размер датасета:", df.shape)

print("\nТипы данных:")
print(df.dtypes)

print("\nПропуски:")
print(df.isna().sum())

print("\nРаспределение target:")
print(df["target"].value_counts(normalize=True))

print("\nРаспределение sex:")
print(df["sex"].value_counts(normalize=True))

print("\nДоля одобрений по sex:")
print(df.groupby("sex")["target"].mean())

print("\nДоля одобрений по age_group:")
print(df.groupby("age_group")["target"].mean())

## 7. Визуализации EDA

Сохраняем графики в `reports/figures/`.

In [ ]:
sns.set_theme(style="whitegrid")

# Распределение целевой переменной.
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="target")
plt.title("Target distribution")
plt.xlabel("Target: 1 = approved, 0 = rejected")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "target_distribution.png", dpi=150)
plt.show()

# Распределение sensitive attribute.
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="sex")
plt.title("Sensitive attribute distribution: sex")
plt.xlabel("Sex")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "group_distribution.png", dpi=150)
plt.show()

# Доля положительного класса по группам.
approval_by_sex = df.groupby("sex", as_index=False)["target"].mean()

plt.figure(figsize=(6, 4))
sns.barplot(data=approval_by_sex, x="sex", y="target")
plt.title("Approval rate by sex")
plt.xlabel("Sex")
plt.ylabel("Approval rate")
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "approval_rate_by_sex.png", dpi=150)
plt.show()

## 8. Выбор sensitive attribute

Для аудита выбран sensitive attribute `sex`.

| Параметр | Значение |
|---|---|
| Sensitive attribute | `sex` |
| Privileged group | `Male` |
| Unprivileged group | `Female` |
| Обоснование | Возможный риск гендерного bias при кредитном скоринге |

Sensitive attribute будет исключён из входных признаков модели.  
Однако bias может сохраняться через proxy features: `income`, `employment_years`, `education`, `region`, `credit_history_years`.

In [ ]:
SENSITIVE_ATTRIBUTE = "sex"
PRIVILEGED_GROUP = "Male"
UNPRIVILEGED_GROUP = "Female"
TARGET_COLUMN = "target"

print("Sensitive attribute:", SENSITIVE_ATTRIBUTE)
print("Privileged group:", PRIVILEGED_GROUP)
print("Unprivileged group:", UNPRIVILEGED_GROUP)

## 9. Предобработка данных

Исключаем из признаков:

- `target`;
- `sex`, так как это sensitive attribute;
- `age_group`, так как он напрямую получен из возраста и используется только для анализа.

При этом `age` остаётся в признаках как обычный числовой признак.  
В реальном проекте это решение нужно отдельно обосновывать.

In [ ]:
excluded_columns = ["target", "sex", "age_group"]

X = df.drop(columns=excluded_columns)
y = df[TARGET_COLUMN]
sensitive_features = df[SENSITIVE_ATTRIBUTE]

categorical_features = ["education", "region"]
numeric_features = [
    "age",
    "income",
    "loan_amount",
    "employment_years",
    "credit_history_years",
    "debt_to_income",
]

X_train, X_test, y_train, y_test, s_train, s_test = train_test_split(
    X,
    y,
    sensitive_features,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("\nSensitive train distribution:")
print(s_train.value_counts(normalize=True))

## 10. Baseline-модель

Используется `LogisticRegression` в составе `Pipeline`.

Preprocessing:

- числовые признаки масштабируются через `StandardScaler`;
- категориальные признаки кодируются через `OneHotEncoder`.

Модель является интерпретируемой и подходит для базового аудита.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

baseline_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]
)

baseline_model.fit(X_train, y_train)

y_pred = baseline_model.predict(X_test)
y_proba = baseline_model.predict_proba(X_test)[:, 1]

print("Baseline-модель обучена.")

## 11. Стандартные метрики качества

Рассчитываем:

- Accuracy;
- Precision;
- Recall;
- F1-score;
- ROC-AUC;
- Confusion matrix.

In [ ]:
def compute_quality_metrics(y_true, y_pred, y_proba=None) -> dict:
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
    }

    if y_proba is not None:
        metrics["roc_auc"] = roc_auc_score(y_true, y_proba)

    return metrics


baseline_quality = compute_quality_metrics(y_test, y_pred, y_proba)

print("Baseline quality metrics:")
for metric, value in baseline_quality.items():
    print(f"{metric}: {value:.4f}")

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

## 12. Групповой анализ качества

Оцениваем модель отдельно для групп `Male` и `Female`.

Для каждой группы рассчитываются:

- count;
- selection rate;
- accuracy;
- precision;
- recall;
- FPR;
- FNR;
- TPR;
- TNR.

In [ ]:
def safe_divide(numerator: float, denominator: float) -> float:
    return float(numerator / denominator) if denominator != 0 else 0.0


def group_metrics_table(y_true, y_pred, sensitive) -> pd.DataFrame:
    rows = []

    y_true_series = pd.Series(y_true).reset_index(drop=True)
    y_pred_series = pd.Series(y_pred).reset_index(drop=True)
    sensitive_series = pd.Series(sensitive).reset_index(drop=True)

    for group in sorted(sensitive_series.unique()):
        mask = sensitive_series == group

        yt = y_true_series[mask]
        yp = y_pred_series[mask]

        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()

        rows.append(
            {
                "group": group,
                "count": int(mask.sum()),
                "selection_rate": selection_rate(yt, yp),
                "accuracy": accuracy_score(yt, yp),
                "precision": precision_score(yt, yp, zero_division=0),
                "recall": recall_score(yt, yp, zero_division=0),
                "fpr": safe_divide(fp, fp + tn),
                "fnr": safe_divide(fn, fn + tp),
                "tpr": safe_divide(tp, tp + fn),
                "tnr": safe_divide(tn, tn + fp),
                "tp": int(tp),
                "fp": int(fp),
                "tn": int(tn),
                "fn": int(fn),
            }
        )

    return pd.DataFrame(rows)


baseline_group_metrics = group_metrics_table(y_test, y_pred, s_test)
baseline_group_metrics.to_csv(REPORTS_DIR / "baseline_group_metrics.csv", index=False)

baseline_group_metrics

## 13. MetricFrame Fairlearn

`MetricFrame` позволяет компактно сравнить метрики по группам.

In [ ]:
fairlearn_metrics = {
    "accuracy": accuracy_score,
    "precision": lambda yt, yp: precision_score(yt, yp, zero_division=0),
    "recall": lambda yt, yp: recall_score(yt, yp, zero_division=0),
    "selection_rate": selection_rate,
    "false_positive_rate": false_positive_rate,
    "false_negative_rate": false_negative_rate,
    "true_positive_rate": true_positive_rate,
}

metric_frame = MetricFrame(
    metrics=fairlearn_metrics,
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=s_test,
)

print("Overall:")
print(metric_frame.overall)

print("\nBy group:")
print(metric_frame.by_group)

## 14. Fairness-метрики baseline-модели

Рассчитываем обязательные fairness-метрики:

- Selection Rate;
- Demographic Parity Difference;
- Demographic Parity Ratio;
- Equalized Odds Difference;
- Equal Opportunity Difference;
- False Positive Rate Difference;
- False Negative Rate Difference.

In [ ]:
def compute_fairness_metrics(y_true, y_pred, sensitive) -> dict:
    mf = MetricFrame(
        metrics={
            "tpr": true_positive_rate,
            "fpr": false_positive_rate,
            "fnr": false_negative_rate,
        },
        y_true=y_true,
        y_pred=y_pred,
        sensitive_features=sensitive,
    )

    by_group = mf.by_group

    fairness = {
        "selection_rate": selection_rate(y_true, y_pred),
        "demographic_parity_difference": demographic_parity_difference(
            y_true,
            y_pred,
            sensitive_features=sensitive,
        ),
        "demographic_parity_ratio": demographic_parity_ratio(
            y_true,
            y_pred,
            sensitive_features=sensitive,
        ),
        "equalized_odds_difference": equalized_odds_difference(
            y_true,
            y_pred,
            sensitive_features=sensitive,
        ),
        "equal_opportunity_difference": float(by_group["tpr"].max() - by_group["tpr"].min()),
        "false_positive_rate_difference": float(by_group["fpr"].max() - by_group["fpr"].min()),
        "false_negative_rate_difference": float(by_group["fnr"].max() - by_group["fnr"].min()),
    }

    return fairness


baseline_fairness = compute_fairness_metrics(y_test, y_pred, s_test)

print("Baseline fairness metrics:")
for metric, value in baseline_fairness.items():
    print(f"{metric}: {value:.4f}")

## 15. Визуализация fairness-метрик baseline

Сохраняем график selection rate по группам.

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(
    data=baseline_group_metrics,
    x="group",
    y="selection_rate",
)
plt.title("Baseline: selection rate by group")
plt.xlabel("Group")
plt.ylabel("Selection rate")
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fairness_metrics_baseline.png", dpi=150)
plt.show()

## 16. Интерпретация baseline-аудита

На этом этапе нужно сделать содержательный вывод:

- какая группа получает больше положительных решений;
- есть ли разница в recall;
- есть ли разница в FPR/FNR;
- может ли это указывать на bias;
- какие proxy features могут поддерживать bias.

In [ ]:
baseline_interpretation = f'''
Baseline-аудит показывает следующие результаты:

- Общая accuracy: {baseline_quality["accuracy"]:.3f}
- Общий recall: {baseline_quality["recall"]:.3f}
- Demographic Parity Difference: {baseline_fairness["demographic_parity_difference"]:.3f}
- Demographic Parity Ratio: {baseline_fairness["demographic_parity_ratio"]:.3f}
- Equalized Odds Difference: {baseline_fairness["equalized_odds_difference"]:.3f}
- Equal Opportunity Difference: {baseline_fairness["equal_opportunity_difference"]:.3f}

Если значения demographic parity difference и equalized odds difference заметно отличаются от 0,
это означает, что группы получают разные доли положительных решений и/или разные ошибки.

Так как sensitive attribute sex был исключён из входных признаков,
потенциальный bias может сохраняться через proxy features:
income, employment_years, education, region, credit_history_years.
'''

print(baseline_interpretation)

## 17. Mitigation bias

Используем post-processing метод `ThresholdOptimizer` из Fairlearn.

Ограничение:

```text
constraints="demographic_parity"
```

Это означает, что метод пытается уменьшить различия в доле положительных решений между группами.

Важно: в реальной системе изменение threshold по группам требует юридической, этической и предметной экспертизы.

In [ ]:
threshold_optimizer = ThresholdOptimizer(
    estimator=baseline_model,
    constraints="demographic_parity",
    predict_method="predict_proba",
    prefit=True,
)

threshold_optimizer.fit(
    X_train,
    y_train,
    sensitive_features=s_train,
)

y_pred_mitigated = threshold_optimizer.predict(
    X_test,
    sensitive_features=s_test,
)

# Для post-processing модели вероятности в том же виде могут быть недоступны,
# поэтому ROC-AUC для mitigated-модели не рассчитываем.
mitigated_quality = compute_quality_metrics(y_test, y_pred_mitigated)
mitigated_fairness = compute_fairness_metrics(y_test, y_pred_mitigated, s_test)
mitigated_group_metrics = group_metrics_table(y_test, y_pred_mitigated, s_test)

mitigated_group_metrics.to_csv(REPORTS_DIR / "mitigated_group_metrics.csv", index=False)

print("Mitigation выполнен.")

## 18. Метрики после mitigation

In [ ]:
print("Mitigated quality metrics:")
for metric, value in mitigated_quality.items():
    print(f"{metric}: {value:.4f}")

print("\nMitigated fairness metrics:")
for metric, value in mitigated_fairness.items():
    print(f"{metric}: {value:.4f}")

print("\nMitigated group metrics:")
mitigated_group_metrics

## 19. Сравнение baseline и mitigated-модели

Сравниваем метрики качества и fairness-метрики до и после mitigation.

In [ ]:
comparison_rows = []

all_metric_names = sorted(set(baseline_quality.keys()) | set(mitigated_quality.keys()))

for metric in all_metric_names:
    baseline_value = baseline_quality.get(metric, np.nan)
    mitigated_value = mitigated_quality.get(metric, np.nan)

    comparison_rows.append(
        {
            "metric": metric,
            "baseline": baseline_value,
            "mitigated": mitigated_value,
            "change": (
                mitigated_value - baseline_value
                if pd.notna(baseline_value) and pd.notna(mitigated_value)
                else np.nan
            ),
            "type": "quality",
        }
    )

for metric in baseline_fairness.keys():
    baseline_value = baseline_fairness[metric]
    mitigated_value = mitigated_fairness[metric]

    comparison_rows.append(
        {
            "metric": metric,
            "baseline": baseline_value,
            "mitigated": mitigated_value,
            "change": mitigated_value - baseline_value,
            "type": "fairness",
        }
    )

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(REPORTS_DIR / "baseline_vs_mitigated_metrics.csv", index=False)

comparison_df

## 20. Визуализация после mitigation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

sns.barplot(
    data=baseline_group_metrics,
    x="group",
    y="selection_rate",
    ax=axes[0],
)
axes[0].set_title("Baseline selection rate")
axes[0].set_xlabel("Group")
axes[0].set_ylabel("Selection rate")
axes[0].set_ylim(0, 1)

sns.barplot(
    data=mitigated_group_metrics,
    x="group",
    y="selection_rate",
    ax=axes[1],
)
axes[1].set_title("Mitigated selection rate")
axes[1].set_xlabel("Group")
axes[1].set_ylabel("Selection rate")
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "fairness_metrics_mitigated.png", dpi=150)
plt.show()

## 21. Сохранение вспомогательных Python-модулей

Создадим файлы в `src/`, чтобы проект соответствовал ожидаемой структуре лабораторной работы.

In [ ]:
Path("src/preprocess.py").write_text(r'''
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def build_preprocessor(numeric_features, categorical_features):
    # Создаёт preprocessing pipeline для числовых и категориальных признаков.
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_features),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ]
    )
'''.strip() + "\n", encoding="utf-8")

Path("src/train.py").write_text(r'''
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


def build_baseline_model(preprocessor, random_state=42):
    # Создаёт baseline-модель LogisticRegression.
    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", LogisticRegression(max_iter=1000, random_state=random_state)),
        ]
    )
'''.strip() + "\n", encoding="utf-8")

Path("src/evaluate.py").write_text(r'''
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score


def compute_quality_metrics(y_true, y_pred, y_proba=None):
    # Рассчитывает стандартные метрики качества классификации.
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
    }

    if y_proba is not None:
        metrics["roc_auc"] = roc_auc_score(y_true, y_proba)

    return metrics
'''.strip() + "\n", encoding="utf-8")

Path("src/fairness_metrics.py").write_text(r'''
from fairlearn.metrics import (
    MetricFrame,
    selection_rate,
    demographic_parity_difference,
    demographic_parity_ratio,
    equalized_odds_difference,
    true_positive_rate,
    false_positive_rate,
    false_negative_rate,
)


def compute_fairness_metrics(y_true, y_pred, sensitive):
    # Рассчитывает fairness-метрики для binary classification.
    mf = MetricFrame(
        metrics={
            "tpr": true_positive_rate,
            "fpr": false_positive_rate,
            "fnr": false_negative_rate,
        },
        y_true=y_true,
        y_pred=y_pred,
        sensitive_features=sensitive,
    )

    by_group = mf.by_group

    return {
        "selection_rate": selection_rate(y_true, y_pred),
        "demographic_parity_difference": demographic_parity_difference(
            y_true,
            y_pred,
            sensitive_features=sensitive,
        ),
        "demographic_parity_ratio": demographic_parity_ratio(
            y_true,
            y_pred,
            sensitive_features=sensitive,
        ),
        "equalized_odds_difference": equalized_odds_difference(
            y_true,
            y_pred,
            sensitive_features=sensitive,
        ),
        "equal_opportunity_difference": float(by_group["tpr"].max() - by_group["tpr"].min()),
        "false_positive_rate_difference": float(by_group["fpr"].max() - by_group["fpr"].min()),
        "false_negative_rate_difference": float(by_group["fnr"].max() - by_group["fnr"].min()),
    }
'''.strip() + "\n", encoding="utf-8")

print("Файлы src созданы.")

## 22. Вспомогательные функции для Markdown-таблиц

Чтобы не зависеть от пакета `tabulate`, создаём собственную простую функцию преобразования DataFrame в Markdown-таблицу.

In [ ]:
def format_metric_table(metrics: dict) -> str:
    lines = ["| Metric | Value |", "|---|---:|"]

    for key, value in metrics.items():
        if isinstance(value, float):
            value_text = f"{value:.4f}"
        else:
            value_text = str(value)

        lines.append(f"| {key} | {value_text} |")

    return "\n".join(lines)


def dataframe_to_markdown_table(dataframe: pd.DataFrame) -> str:
    df_copy = dataframe.copy()

    for column in df_copy.columns:
        if pd.api.types.is_float_dtype(df_copy[column]):
            df_copy[column] = df_copy[column].map(lambda value: f"{value:.4f}")

    columns = list(df_copy.columns)

    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join(["---"] * len(columns)) + " |"

    rows = []
    for _, row in df_copy.iterrows():
        rows.append("| " + " | ".join(str(row[column]) for column in columns) + " |")

    return "\n".join([header, separator] + rows)


print("Функции для Markdown-таблиц созданы.")

## 23. Генерация Model Card

Model Card сохраняется в:

```text
reports/model_card.md
```

In [ ]:
model_card = f'''# Model Card

## Model Details

- Model name: Demo Credit Scoring Logistic Regression
- Version: 1.0.0
- Date: auto-generated
- Author: student
- Model type: Binary classifier
- Library / framework: scikit-learn, Fairlearn

## Intended Use

- Primary intended use: учебная демонстрация кредитного скоринга и fairness-аудита.
- Intended users: студенты и преподаватели курса.
- Out-of-scope use cases: реальное принятие кредитных решений, production scoring, юридически значимые решения.

## Training Data

- Dataset name: Synthetic Credit Scoring Dataset
- Dataset source: generated programmatically
- Dataset size: {df.shape[0]} rows, {df.shape[1]} columns
- Data collection period: not applicable
- Sensitive attributes: sex
- Known limitations: synthetic data, intentionally embedded historical bias.

## Target

- `target = 1`: loan application approved.
- `target = 0`: loan application rejected.

## Sensitive Attribute

| Sensitive attribute | Privileged group | Unprivileged group | Rationale |
|---|---|---|---|
| sex | Male | Female | Возможный риск гендерного bias при кредитном скоринге |

## Features Used

{", ".join(X.columns)}

## Features Excluded

- target
- sex
- age_group

Sensitive attribute `sex` excluded from model input, but bias may remain through proxy features.

## Evaluation Data

- Test split: 20%
- Evaluation method: train/test split with stratification by target
- Groups evaluated: Male, Female

## Metrics

### Overall Performance — Baseline

{format_metric_table(baseline_quality)}

### Overall Performance — After Mitigation

{format_metric_table(mitigated_quality)}

### Fairness Metrics — Baseline

{format_metric_table(baseline_fairness)}

### Fairness Metrics — After Mitigation

{format_metric_table(mitigated_fairness)}

## Evaluation by Group — Baseline

{dataframe_to_markdown_table(baseline_group_metrics)}

## Evaluation by Group — After Mitigation

{dataframe_to_markdown_table(mitigated_group_metrics)}

## Ethical Considerations

- Potential harms: необоснованный отказ в кредите отдельным группам.
- Groups at risk: группы, получающие меньше положительных решений или больше false negative.
- Bias detected: определяется по demographic parity difference, equalized odds difference и групповым метрикам.
- Mitigation applied: Fairlearn ThresholdOptimizer with demographic parity constraint.

## Limitations

- Data limitations: данные синтетические и не отражают полностью реальные социальные процессы.
- Model limitations: Logistic Regression является простой baseline-моделью.
- Fairness limitations: оптимизация одной fairness-метрики может ухудшать другие метрики.
- Deployment limitations: модель не предназначена для production.

## Recommendations

- Recommended use: только учебный fairness-аудит.
- Monitoring requirements: мониторинг quality, fairness, drift, prediction distribution.
- Human oversight: требуется для любых решений, влияющих на пользователей.
- Retraining requirements: при изменении распределения данных и ухудшении fairness-метрик.

## Prohibited Uses

- Реальное кредитное решение.
- Автоматическое принятие юридически значимых решений.
- Использование без дополнительной проверки качества, fairness, устойчивости и правовых требований.
'''

Path("reports/model_card.md").write_text(model_card, encoding="utf-8")

print("reports/model_card.md создан.")

## 24. Генерация отчёта об аудите bias

Отчёт сохраняется в:

```text
reports/bias_audit_report.md
```

In [ ]:
bias_audit_report = f'''# Bias Audit Report

## 1. Титульная информация

- Лабораторная работа: Анализ bias и аудит модели
- ФИО студента:
- Группа:
- Дата выполнения:
- Датасет: Synthetic Credit Scoring Dataset
- Библиотека fairness-аудита: Fairlearn

## 2. Описание задачи

Модель решает задачу кредитного скоринга.

Положительный класс означает одобрение заявки.  
False negative может привести к необоснованному отказу добросовестному клиенту.  
False positive может привести к финансовым потерям организации.

Задача является чувствительной с точки зрения fairness, так как решение может влиять на доступ пользователя к финансовым услугам.

## 3. Описание данных

| Раздел | Значение |
|---|---|
| Источник данных | Synthetic generated dataset |
| Размер данных | {df.shape[0]} строк, {df.shape[1]} признаков |
| Целевая переменная | target |
| Sensitive attribute | sex |
| Privileged group | Male |
| Unprivileged group | Female |
| Пропуски | {int(df.isna().sum().sum())} |
| Тип задачи | Binary classification |

## 4. Baseline-модель

Алгоритм: Logistic Regression.

Модель выбрана как простая и интерпретируемая baseline-модель.

### Метрики качества

{format_metric_table(baseline_quality)}

## 5. Групповой анализ

{dataframe_to_markdown_table(baseline_group_metrics)}

## 6. Fairness-метрики baseline

{format_metric_table(baseline_fairness)}

## 7. Анализ причин bias

Возможные причины bias:

1. Исторический bias в данных: метки были сгенерированы так, что группа Female имела меньшую вероятность положительного решения.
2. Proxy features: income, education, region, employment_years могут коррелировать с sensitive attribute.
3. Дисбаланс групп и различия распределений признаков между группами.
4. Некорректный единый threshold классификации для всех групп.
5. Label bias: целевая переменная отражает исторические решения, а не объективную кредитоспособность.

## 8. Mitigation bias

Метод: ThresholdOptimizer.

Тип метода: Post-processing.

Ограничение: demographic parity.

Метод выбран, потому что он позволяет уменьшить различие selection rate между группами без переобучения исходной модели.

## 9. Сравнение до и после mitigation

{dataframe_to_markdown_table(comparison_df)}

## 10. Выводы

- Был проведён fairness-аудит baseline-модели.
- Рассчитаны стандартные метрики качества.
- Выполнен групповой анализ по sensitive attribute `sex`.
- Рассчитаны fairness-метрики.
- Применён mitigation bias через ThresholdOptimizer.
- Подготовлен Model Card.

Перед production deployment потребуются:

- аудит на реальных данных;
- юридическая и этическая экспертиза;
- анализ нескольких sensitive attributes;
- анализ intersectional fairness;
- мониторинг fairness и drift;
- human oversight;
- регулярное обновление Model Card.
'''

Path("reports/bias_audit_report.md").write_text(bias_audit_report, encoding="utf-8")

print("reports/bias_audit_report.md создан.")

## 25. README.md

Создаём инструкцию по воспроизведению эксперимента.

In [ ]:
Path("README.md").write_text(
    '''# Лабораторная работа 8.2
## Анализ bias и аудит модели

Проект содержит полный pipeline fairness-аудита ML-модели.

## Структура проекта

```text
.
├── data/
│   ├── README.md
│   └── sample_data.csv
├── notebooks/
├── src/
│   ├── preprocess.py
│   ├── train.py
│   ├── evaluate.py
│   └── fairness_metrics.py
├── reports/
│   ├── model_card.md
│   ├── bias_audit_report.md
│   ├── baseline_group_metrics.csv
│   ├── mitigated_group_metrics.csv
│   ├── baseline_vs_mitigated_metrics.csv
│   └── figures/
├── requirements.txt
└── README.md
```

## Установка зависимостей

```bash
pip install -r requirements.txt
```

## Запуск

Открыть ноутбук:

```bash
jupyter notebook lab_8_2_bias_audit.ipynb
```

и выполнить ячейки сверху вниз.

## Что реализовано

- генерация синтетического датасета;
- выбор sensitive attribute;
- EDA;
- preprocessing;
- baseline-модель Logistic Regression;
- стандартные метрики качества;
- групповой анализ качества модели;
- fairness-метрики через Fairlearn;
- mitigation bias через ThresholdOptimizer;
- сравнение baseline и mitigated;
- Model Card;
- отчёт bias audit.

## Sensitive attribute

Основной sensitive attribute:

```text
sex: Male / Female
```

## Важное ограничение

Данные синтетические.  
Модель не предназначена для реального кредитного скоринга или production deployment.
''',
    encoding="utf-8",
)

print("README.md создан.")

## 26. Финальная проверка артефактов

Выведем список созданных файлов.

In [ ]:
for path in sorted(Path(".").rglob("*")):
    if path.is_file() and ".ipynb_checkpoints" not in str(path):
        print(path)

## 27. Минимальный чек-лист выполнения

- [x] Датасет загружен и описан.
- [x] Целевая переменная определена.
- [x] Sensitive attribute выбран и обоснован.
- [x] Выполнен EDA.
- [x] Выполнена предобработка данных.
- [x] Обучена baseline-модель.
- [x] Рассчитаны accuracy, precision, recall, F1-score.
- [x] Выполнен анализ качества по группам.
- [x] Рассчитаны fairness-метрики.
- [x] Интерпретированы результаты fairness-аудита.
- [x] Применён mitigation bias.
- [x] Выполнено сравнение до и после mitigation.
- [x] Подготовлен `model_card.md`.
- [x] Сделаны выводы и рекомендации.
- [x] Эксперимент воспроизводится по инструкции из README.